In [2]:
import os
import numpy as np
import pandas as pd
import rasterio
from rasterio.mask import mask
from rasterio.warp import calculate_default_transform, reproject, Resampling
import geopandas as gpd

# ---------------- USER INPUTS ----------------
r_process = r"D:\Phd Research\Final_Raster\Inundation_100yr_Surge+SLR.tif"          # process-based depth raster (m)
r_bathtub = r"D:\Phd Research\Final_Raster\Bathtub_depth_100yr_surge_SLR.tif"       # bathtub depth raster (m)
boundary_zip = r"D:\Phd Research\GIS\Shape\land_Part_area_utm.zip"                  # land boundary polygon (UTM Zone 14N)
FLOOD_MIN = 0.10  # meters; treat anything < FLOOD_MIN as dry
depth_bins = [(0.1, 2.5), (2.5, 6.0), (6.0, 8.5), (8.5, 12.0)]  # note 0.1 lower bound for first bin
TARGET_METRIC_CRS = "EPSG:32614"  # UTM 14N (meters)

# --------------- HELPERS ----------------
def ensure_metric_and_clip(src_path, boundary_gdf):
    """Reproject raster to metric CRS if needed, clip to boundary, return (array, transform, crs)."""
    with rasterio.open(src_path) as src:
        # Reproject boundary to raster CRS (or vice versa) for clipping
        bnd_in_src = boundary_gdf.to_crs(src.crs)

        # Clip (crop) to land boundary
        out_img, out_transform = mask(src, bnd_in_src.geometry, crop=True)
        arr = out_img[0]  # single band
        nodata = src.nodata

        # Apply nodata mask
        if nodata is not None:
            arr = np.where(arr == nodata, np.nan, arr)

        # If raster CRS is geographic (degrees), reproject clipped array to TARGET_METRIC_CRS
        if src.crs.is_geographic:
            dst_crs = TARGET_METRIC_CRS
            transform, width, height = calculate_default_transform(
                src.crs, dst_crs, out_img.shape[2], out_img.shape[1], *rasterio.transform.array_bounds(out_img.shape[1], out_img.shape[2], out_transform)
            )
            dst = np.full((height, width), np.nan, dtype=np.float32)
            reproject(
                source=arr,
                destination=dst,
                src_transform=out_transform,
                src_crs=src.crs,
                dst_transform=transform,
                dst_crs=dst_crs,
                resampling=Resampling.bilinear,
                src_nodata=np.nan,
                dst_nodata=np.nan,
            )
            return dst, transform, rasterio.crs.CRS.from_string(dst_crs)
        else:
            # Already metric
            return arr, out_transform, src.crs

def pixel_area_km2(transform):
    # abs(cell width * cell height) in square meters -> km²
    return abs(transform.a * transform.e) / 1e6

def compute_bins(arr, transform, bins):
    """Return list of (n_cells, area_km2) per bin, ignoring NaNs and applying FLOOD_MIN rule."""
    # Mask out non-flood (arr < FLOOD_MIN) and NaNs
    valid = (~np.isnan(arr)) & (arr >= FLOOD_MIN)
    cell_area = pixel_area_km2(transform)
    stats = []
    for low, high in bins:
        m = valid & (arr >= low) & (arr < high)
        n = int(np.count_nonzero(m))
        area_km2 = n * cell_area
        stats.append((n, area_km2))
    # Also return totals for sanity checks
    total_cells = int(np.count_nonzero(valid))
    total_area = total_cells * cell_area
    return stats, total_cells, total_area

# --------------- MAIN ----------------
# Read land boundary (assumed UTM 14N)
boundary = gpd.read_file(boundary_zip)
if boundary.crs is None:
    raise ValueError("Boundary CRS is None. Please define it (expected EPSG:32614).")

# Process-based
proc_arr, proc_transform, _ = ensure_metric_and_clip(r_process, boundary)
proc_stats, proc_total_cells, proc_total_area = compute_bins(proc_arr, proc_transform, depth_bins)

# Bathtub
bath_arr, bath_transform, _ = ensure_metric_and_clip(r_bathtub, boundary)
bath_stats, bath_total_cells, bath_total_area = compute_bins(bath_arr, bath_transform, depth_bins)

# Build table rows
rows = []
for (low, high), (pn, pa), (bn, ba) in zip(depth_bins, proc_stats, bath_stats):
    rows.append({
        "Depth range (m)": f"{low}-{high}",
        "Number of grid cells - Process-based": pn,
        "Number of grid cells - Bath-tub": bn,
        "Area (km²) - Process-based": round(pa, 2),
        "Area (km²) - Bath-tub": round(ba, 2),
    })

# Optional: add TOTAL row
rows.append({
    "Depth range (m)": "TOTAL (≥ {:.2f})".format(FLOOD_MIN),
    "Number of grid cells - Process-based": proc_total_cells,
    "Number of grid cells - Bath-tub": bath_total_cells,
    "Area (km²) - Process-based": round(proc_total_area, 2),
    "Area (km²) - Bath-tub": round(bath_total_area, 2),
})

df = pd.DataFrame(rows)
print(df.to_string(index=False))

# Save
out_csv = os.path.join(os.path.dirname(r_process), "Depth_Area_Table_process_vs_bathtub.csv")
df.to_csv(out_csv, index=False)
print(f"\nSaved: {out_csv}")


Depth range (m)  Number of grid cells - Process-based  Number of grid cells - Bath-tub  Area (km²) - Process-based  Area (km²) - Bath-tub
        0.1-2.5                                 43183                            66826                     1727.32                2673.04
        2.5-6.0                                152145                           215919                     6085.80                8636.76
        6.0-8.5                                131306                            54843                     5252.24                2193.72
       8.5-12.0                                 19373                                0                      774.92                   0.00
 TOTAL (≥ 0.10)                                347452                           337588                    13898.08               13503.52

Saved: D:\Phd Research\Final_Raster\Depth_Area_Table_process_vs_bathtub.csv


In [3]:
import os
import numpy as np
import pandas as pd
import rasterio
from rasterio.mask import mask
from rasterio.warp import calculate_default_transform, reproject, Resampling
import geopandas as gpd

# ---------------- USER INPUTS ----------------
r_process = r"D:\Phd Research\Final_Raster\100yr_compound_flood_base_stat.tif"          # process-based depth raster (m)
r_bathtub = r"D:\Phd Research\Final_Raster\Bathtub_depth_100yr_surge_SLR.tif"       # bathtub depth raster (m)
boundary_zip = r"D:\Phd Research\GIS\Shape\land_Part_area_utm.zip"                  # land boundary polygon (UTM Zone 14N)
FLOOD_MIN = 0.10  # meters; treat anything < FLOOD_MIN as dry
depth_bins = [(0.1, 2.5), (2.5, 6.0), (6.0, 8.5), (8.5, 12.0)]  # note 0.1 lower bound for first bin
TARGET_METRIC_CRS = "EPSG:32614"  # UTM 14N (meters)

# --------------- HELPERS ----------------
def ensure_metric_and_clip(src_path, boundary_gdf):
    """Reproject raster to metric CRS if needed, clip to boundary, return (array, transform, crs)."""
    with rasterio.open(src_path) as src:
        # Reproject boundary to raster CRS (or vice versa) for clipping
        bnd_in_src = boundary_gdf.to_crs(src.crs)

        # Clip (crop) to land boundary
        out_img, out_transform = mask(src, bnd_in_src.geometry, crop=True)
        arr = out_img[0]  # single band
        nodata = src.nodata

        # Apply nodata mask
        if nodata is not None:
            arr = np.where(arr == nodata, np.nan, arr)

        # If raster CRS is geographic (degrees), reproject clipped array to TARGET_METRIC_CRS
        if src.crs.is_geographic:
            dst_crs = TARGET_METRIC_CRS
            transform, width, height = calculate_default_transform(
                src.crs, dst_crs, out_img.shape[2], out_img.shape[1], *rasterio.transform.array_bounds(out_img.shape[1], out_img.shape[2], out_transform)
            )
            dst = np.full((height, width), np.nan, dtype=np.float32)
            reproject(
                source=arr,
                destination=dst,
                src_transform=out_transform,
                src_crs=src.crs,
                dst_transform=transform,
                dst_crs=dst_crs,
                resampling=Resampling.bilinear,
                src_nodata=np.nan,
                dst_nodata=np.nan,
            )
            return dst, transform, rasterio.crs.CRS.from_string(dst_crs)
        else:
            # Already metric
            return arr, out_transform, src.crs

def pixel_area_km2(transform):
    # abs(cell width * cell height) in square meters -> km²
    return abs(transform.a * transform.e) / 1e6

def compute_bins(arr, transform, bins):
    """Return list of (n_cells, area_km2) per bin, ignoring NaNs and applying FLOOD_MIN rule."""
    # Mask out non-flood (arr < FLOOD_MIN) and NaNs
    valid = (~np.isnan(arr)) & (arr >= FLOOD_MIN)
    cell_area = pixel_area_km2(transform)
    stats = []
    for low, high in bins:
        m = valid & (arr >= low) & (arr < high)
        n = int(np.count_nonzero(m))
        area_km2 = n * cell_area
        stats.append((n, area_km2))
    # Also return totals for sanity checks
    total_cells = int(np.count_nonzero(valid))
    total_area = total_cells * cell_area
    return stats, total_cells, total_area

# --------------- MAIN ----------------
# Read land boundary (assumed UTM 14N)
boundary = gpd.read_file(boundary_zip)
if boundary.crs is None:
    raise ValueError("Boundary CRS is None. Please define it (expected EPSG:32614).")

# Process-based
proc_arr, proc_transform, _ = ensure_metric_and_clip(r_process, boundary)
proc_stats, proc_total_cells, proc_total_area = compute_bins(proc_arr, proc_transform, depth_bins)

# Bathtub
bath_arr, bath_transform, _ = ensure_metric_and_clip(r_bathtub, boundary)
bath_stats, bath_total_cells, bath_total_area = compute_bins(bath_arr, bath_transform, depth_bins)

# Build table rows
rows = []
for (low, high), (pn, pa), (bn, ba) in zip(depth_bins, proc_stats, bath_stats):
    rows.append({
        "Depth range (m)": f"{low}-{high}",
        "Number of grid cells - Process-based": pn,
        "Number of grid cells - Bath-tub": bn,
        "Area (km²) - Process-based": round(pa, 2),
        "Area (km²) - Bath-tub": round(ba, 2),
    })

# Optional: add TOTAL row
rows.append({
    "Depth range (m)": "TOTAL (≥ {:.2f})".format(FLOOD_MIN),
    "Number of grid cells - Process-based": proc_total_cells,
    "Number of grid cells - Bath-tub": bath_total_cells,
    "Area (km²) - Process-based": round(proc_total_area, 2),
    "Area (km²) - Bath-tub": round(bath_total_area, 2),
})

df = pd.DataFrame(rows)
print(df.to_string(index=False))

# Save
out_csv = os.path.join(os.path.dirname(r_process), "Depth_Area_Table_process_vs_bathtub.csv")
df.to_csv(out_csv, index=False)
print(f"\nSaved: {out_csv}")


Depth range (m)  Number of grid cells - Process-based  Number of grid cells - Bath-tub  Area (km²) - Process-based  Area (km²) - Bath-tub
        0.1-2.5                                 72907                            66826                     2916.28                2673.04
        2.5-6.0                                202896                           215919                     8115.84                8636.76
        6.0-8.5                                 55753                            54843                     2230.12                2193.72
       8.5-12.0                                 10985                                0                      439.40                   0.00
 TOTAL (≥ 0.10)                                343822                           337588                    13752.88               13503.52

Saved: D:\Phd Research\Final_Raster\Depth_Area_Table_process_vs_bathtub.csv
